In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import defaultdict

## Evaluation Metrics

In [ ]:
def compute_iou(box1, box2):
    """Compute IoU between two boxes in xyxy format."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    if union == 0:
        return 0.0
    return intersection / union

In [ ]:
def compute_precision_recall(tp, fp, num_gt):
    """Compute cumulative precision and recall arrays from TP/FP arrays."""
    tp_cumsum = np.cumsum(tp)
    fp_cumsum = np.cumsum(fp)

    precision = tp_cumsum / (tp_cumsum + fp_cumsum)
    recall = tp_cumsum / num_gt if num_gt > 0 else tp_cumsum
    
    return precision, recall

def compute_f1(precision, recall):
    """Compute F1 score from precision and recall (scalar or array)."""
    denom = precision + recall
    return np.where(denom == 0, 0.0, 2 * precision * recall / denom)

In [ ]:
def compute_ap(tp, fp, num_gt):
    """Compute Average Precision for one class.

    1. Build cumulative precision/recall arrays
    2. Apply precision envelope correction (monotonically decreasing)
    3. Compute AP as the area under the corrected PR curve
    """
    if num_gt == 0:
        return 0.0, np.array([]), np.array([])

    precision, recall = compute_precision_recall(tp, fp, num_gt)

    # Prepend (recall=0, precision=1) for area computation
    recall = np.concatenate(([0.0], recall))
    precision = np.concatenate(([1.0], precision))

    # Precision envelope: traverse from end to start,
    # replace each precision by the max precision at any higher recall
    for i in range(len(precision) - 2, -1, -1):
        precision[i] = max(precision[i], precision[i + 1])

    # Compute AP as sum of rectangular areas
    ap = 0.0
    for i in range(1, len(recall)):
        ap += (recall[i] - recall[i - 1]) * precision[i]

    return ap, precision, recall

In [ ]:
def compute_map(gt_by_image, preds_by_image, all_classes, iou_threshold=0.5):
    """Compute mAP across all classes at a given IoU threshold.
    Returns: mAP (float), per-class results (dict)."""
    per_class = {}

    for cls in all_classes:
        tp, fp, num_gt = match_predictions(gt_by_image, preds_by_image, cls, iou_threshold)
        ap, precision, recall = compute_ap(tp, fp, num_gt)

        total_tp = int(tp.sum())
        total_fp = int(fp.sum())
        final_prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
        final_rec = total_tp / num_gt if num_gt > 0 else 0.0
        f1 = 2 * final_prec * final_rec / (final_prec + final_rec) if (final_prec + final_rec) > 0 else 0.0

        per_class[cls] = {
            "ap": ap,
            "precision": final_prec,
            "recall": final_rec,
            "f1": f1,
            "num_gt": num_gt,
            "num_preds": len(tp),
            "tp": total_tp,
            "fp": total_fp,
            "pr_curve_precision": precision,
            "pr_curve_recall": recall,
        }

    map_val = np.mean([v["ap"] for v in per_class.values()]) if per_class else 0.0
    return map_val, per_class

In [ ]:
def match_predictions(gt_by_image, preds_by_image, class_id, iou_threshold=0.5):
    """Match predictions to ground-truth for a single class.

    1. Collect all predictions for this class across all images
    2. Sort by confidence (descending)
    3. For each prediction, find the best-matching unmatched GT box (highest IoU)
    4. If best IoU >= threshold and GT not yet matched -> TP, else FP

    Returns: tp_list, fp_list, num_gt (total ground-truth boxes for this class)
    """
    all_preds = []
    num_gt = 0
    gt_matched = {}

    for image_id, gt_boxes in gt_by_image.items():
        class_gt = [b for b in gt_boxes if b["class_id"] == class_id]
        num_gt += len(class_gt)
        gt_matched[image_id] = [False] * len(class_gt)

    for image_id, pred_boxes in preds_by_image.items():
        for pred in pred_boxes:
            if pred["class_id"] == class_id:
                all_preds.append((image_id, pred))

    all_preds.sort(key=lambda x: x[1].get("confidence", 0), reverse=True)

    tp = np.zeros(len(all_preds))
    fp = np.zeros(len(all_preds))

    for i, (image_id, pred) in enumerate(all_preds):
        pred_box = yolo_to_xyxy(pred["bbox"])

        class_gt = [b for b in gt_by_image.get(image_id, []) if b["class_id"] == class_id]

        best_iou = 0
        best_gt_idx = -1
        for j, gt in enumerate(class_gt):
            gt_box = yolo_to_xyxy(gt["bbox"])
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = j

        if best_iou >= iou_threshold and not gt_matched[image_id][best_gt_idx]:
            tp[i] = 1
            gt_matched[image_id][best_gt_idx] = True
        else:
            fp[i] = 1

    return tp, fp, num_gt